In [15]:
import cv2
import cvzone
import time
from pynput.keyboard import Controller
from cvzone.HandTrackingModule import HandDetector

# Webcam Setup
cap = cv2.VideoCapture(0)
cap.set(3, 1280)
cap.set(4, 720)

# Hand Detector
detector = HandDetector(detectionCon=0.8)

# Keyboard Controller
keyboard = Controller()

finalText = ""
lastClickTime = 0
delay = 0.8

# Button Class
class Button:
    def __init__(self, pos, text, size=(85, 85)):
        self.pos = pos
        self.size = size
        self.text = text

# Draw Keyboard
def drawAll(img, buttonList):
    for button in buttonList:
        x, y = button.pos
        w, h = button.size

        cv2.rectangle(img, (x, y), (x+w, y+h), (50, 50, 50), cv2.FILLED)
        cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 255), 3)

        cv2.putText(img, button.text, (x+15, y+55),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,0), 3)
    return img

# Keyboard Layout
keys = [
    ["Q","W","E","R","T","Y","U","I","O","P"],
    ["A","S","D","F","G","H","J","K","L"],
    ["Z","X","C","V","B","N","M"],
    ["SPACE","BACKSPACE"]
]

# Create Buttons
buttonList = []

for i in range(len(keys)):
    for j, key in enumerate(keys[i]):

        if key == "SPACE":
            buttonList.append(Button([150, 350], key, size=(400, 85)))

        elif key == "BACKSPACE":
            buttonList.append(Button([600, 350], key, size=(350, 85)))

        else:
            buttonList.append(Button([100*j+50, 100*i+50], key))

# Main Loop
while True:

    success, img = cap.read()
    img = cv2.flip(img, 1)

    frame = img.copy()
    h, w, _ = frame.shape

    # ---------------- FACE CAMERA (RIGHT MIDDLE) ----------------
    cam_w, cam_h = 320, 180
    face_preview = cv2.resize(img, (cam_w, cam_h))

    x_start = w - cam_w - 10
    y_start = (h // 2) - (cam_h // 2)

    # Border
    cv2.rectangle(frame,
                  (x_start, y_start),
                  (x_start + cam_w, y_start + cam_h),
                  (255, 0, 255), 2)

    # Place camera
    frame[y_start:y_start+cam_h, x_start:x_start+cam_w] = face_preview
    # ------------------------------------------------------------

    # Detect Hands
    hands, frame = detector.findHands(frame)

    # Draw Keyboard
    frame = drawAll(frame, buttonList)

    if hands:

        hand1 = hands[0]
        lmList = hand1["lmList"]

        x1, y1 = lmList[8][0], lmList[8][1]  # index fingertip

        for button in buttonList:

            x, y = button.pos
            w_b, h_b = button.size

            # Hover detection
            if x < x1 < x+w_b and y < y1 < y+h_b:

                cv2.rectangle(frame, button.pos, (x+w_b, y+h_b), (0, 255, 255), cv2.FILLED)
                cv2.putText(frame, button.text, (x+15, y+55),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,0), 3)

                # ---------------- PINCH CLICK (Thumb + Index) ----------------
                length, info, frame = detector.findDistance(
                    lmList[8][0:2],   # index finger
                    lmList[4][0:2],   # thumb finger
                    frame
                )

                currentTime = time.time()

                if length < 40 and currentTime - lastClickTime > delay:

                    if button.text == "SPACE":
                        finalText += " "

                    elif button.text == "BACKSPACE":
                        finalText = finalText[:-1]

                    else:
                        keyboard.press(button.text)
                        finalText += button.text

                    lastClickTime = currentTime

                    # Click animation
                    cv2.rectangle(frame, button.pos, (x+w_b, y+h_b), (0,255,0), cv2.FILLED)
                    cv2.putText(frame, button.text, (x+15, y+55),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,0), 3)

    # ---------------- TEXT DISPLAY BOX ----------------
    cv2.rectangle(frame, (40, 500), (1240, 700), (20,20,20), cv2.FILLED)
    cv2.rectangle(frame, (40, 500), (1240, 700), (255,0,255), 3)

    displayText = finalText[-80:]
    lines = [displayText[i:i+25] for i in range(0, len(displayText), 25)]

    y_pos = 560
    for line in lines:
        cv2.putText(frame, line, (70, y_pos),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 3)
        y_pos += 50

    # Show output
    cv2.imshow("Virtual Keyboard + Camera UI", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()